In [49]:
import numpy as np
import pandas as pd


In [50]:
df = pd.read_csv("marketing_campaign_data_messy.csv")
print(f"DataFrame Create Successfully\nRows: {df.shape[0]}, columns: {df.shape[1]}")

DataFrame Create Successfully
Rows: 2020, columns: 12


In [51]:
df.head()

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA


## Cleaning Hearders


In [52]:
print(df.columns.to_list())
df.columns = df.columns.str.strip().str.lower().str.replace(" ","_")
print("Fixed applied successfully")
print(df.columns.to_list())

[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign_Tag']
Fixed applied successfully
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks', 'campaign_tag']


### Fixing

In [53]:
dirty_spend_mask = df["spend"].astype(str).str.contains(r"\$")
print(df.loc[dirty_spend_mask,["campaign_id", "spend"]].head())
df["spend"] = df["spend"].astype(str).str.replace(r'[^\d.-]','',regex=True)
print("Fixed applied")
df["spend"] = pd.to_numeric(df["spend"])
print(df.loc[dirty_spend_mask,["campaign_id", "spend"]].head())


   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22
31   CMP-00032  $2759.35
32   CMP-00033  $2393.02
Fixed applied
   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22
31   CMP-00032  2759.35
32   CMP-00033  2393.02


## Handling unique

In [54]:
print(df["channel"].unique())

clean_map = {
    "TikTok" : "TikTok",
    "Tik_Tok" : "TikTok",

    "Facebook" : "Facebook",
    "Facebok" : "Facebook",

    "Email" : "Email",
    "E-mail" : "Email",

    "Instagram" : "Instagram",
    "Insta_gram" : "Instagram",
    "Google Ads" : "Google_Ads",
    "Gogle" : "Google_Ads",
    "N/A" : np.nan,
}

df["channel"] = df["channel"].replace(clean_map)
print(df["channel"].unique())


<StringArray>
[    'TikTok',   'Facebook',      'Email',  'Instagram', 'Google Ads',
     'E-mail',          nan,      'Gogle',    'Tik_Tok',    'Facebok',
 'Insta_gram']
Length: 11, dtype: str
<StringArray>
['TikTok', 'Facebook', 'Email', 'Instagram', 'Google_Ads', nan]
Length: 6, dtype: str


In [55]:
#Mixed Booleans

print(df["active"].unique())

bool_map = {
    "Y" : True,
    "True" : True,
    "1" : True,

    '0' : False,
    "False" : False,
    'No' : False,
}

df["active"] = df["active"].map(bool_map).fillna(False).astype(bool)
print(df["active"].unique())

<StringArray>
['Y', '0', 'No', 'True', 'Yes', '1', 'False']
Length: 7, dtype: str
[ True False]


# DateParsing

In [59]:
print(df["start_date"].dtype)

df["start_date"] = pd.to_datetime(df["start_date"], errors='coerce')
df["end_date"] = pd.to_datetime(df["end_date"], errors='coerce')

print(df["start_date"].dtype)

str
datetime64[us]


In [61]:
df = df.loc[:, ~df.columns.duplicated(keep="first")]
df

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,False,FA
...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,True,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google_Ads,20097,897,1641.00,162.0,False,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,False,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,False,FA


In [62]:
impossible_mask = df["clicks"] > df["impressions"]
print(df.loc[impossible_mask,["campaign_id","clicks" ,"impressions"]].head())

Empty DataFrame
Columns: [campaign_id, clicks, impressions]
Index: []


In [65]:
timetravel_mask = df["start_date"] > df["end_date"]
print(df.loc[timetravel_mask,["campaign_id","start_date","end_date"]].head())

df.loc[timetravel_mask, "end_date"] = df.loc[timetravel_mask, "start_date"] + pd.Timedelta(days=30)

print("Fixed Applied")
print(df.loc[timetravel_mask,["campaign_id","start_date","end_date"]].head())

    campaign_id start_date   end_date
23    CMP-00024 2023-05-06 2023-05-01
54    CMP-00055 2023-09-01 2023-08-27
71    CMP-00072 2023-02-01 2023-01-27
156   CMP-00157 2023-12-06 2023-12-01
200   CMP-00201 2023-01-11 2023-01-06
Fixed Applied
    campaign_id start_date   end_date
23    CMP-00024 2023-05-06 2023-06-05
54    CMP-00055 2023-09-01 2023-10-01
71    CMP-00072 2023-02-01 2023-03-03
156   CMP-00157 2023-12-06 2024-01-05
200   CMP-00201 2023-01-11 2023-02-10


In [68]:

Q1 = df["spend"].quantile(0.25)
Q3 = df["spend"].quantile(0.75)

IQR = Q3 - Q1

upper_limit = Q3 + (3 * IQR)

outlier_mask = df["spend"] > upper_limit
print(df.loc[outlier_mask, ["campaign_id", "spend"]])

print("Applied Fixed")
df.loc[outlier_mask, "spend"] = upper_limit
print(df.loc[outlier_mask, ["campaign_id", "spend"]])

     campaign_id      spend
789    CMP-00790  500000.00
1443   CMP-01444    8921.51
1460   CMP-01461  500000.00
1718   CMP-01719  500000.00
1754   CMP-01755  500000.00
1781   CMP-01782  500000.00
Applied Fixed
     campaign_id      spend
789    CMP-00790  8603.5375
1443   CMP-01444  8603.5375
1460   CMP-01461  8603.5375
1718   CMP-01719  8603.5375
1754   CMP-01755  8603.5375
1781   CMP-01782  8603.5375


In [72]:
df["campaign_name"].head()
df["season"] = df["campaign_name"].str.extract(r'Q\d_([^_]+)_')

print(df[["campaign_name","season"]].head())

              campaign_name       season
0       Q4_Summer_CMP-00001       Summer
1       Q1_Launch_CMP-00002       Launch
2       Q3_Winter_CMP-00003       Winter
3  Q1_BlackFriday_CMP-00004  BlackFriday
4       Q2_Winter_CMP-00005       Winter


In [73]:
df.to_csv("cleaned_csv.csv", index=False)